In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — JetBot Data Collection: Setup, Helpers, UI
# ─────────────────────────────────────────────────────────────────────────────

# ── Imports ──────────────────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import traitlets
import threading
import time
from pathlib import Path

try:
    from jetbot import Robot, Camera, bgr8_to_jpeg, Heartbeat
    JETBOT_AVAILABLE = True
except ImportError:
    JETBOT_AVAILABLE = False
    print("⚠️  jetbot not found — running in simulation mode (no motors/camera).")

# ── Tunable Parameters ────────────────────────────────────────────────────────
BURST_SPEED    = 0.4   # motor power 0.0–1.0  ← tune this
BURST_DURATION = 0.3   # seconds per burst     ← tune this
SWAP_MOTORS    = True  # True if left/right are physically reversed

# ── Global State ──────────────────────────────────────────────────────────────
_camera        = None
_camera_link   = None
_heartbeat     = None
_robot         = None
_counts        = {'left': 0, 'forward': 0, 'right': 0}
_run_dirs      = {}
_run_ready     = False
# _burst_cancel  = threading.Event()

# ── Cleanup (safe to call any time / on re-run) ───────────────────────────────
def cleanup():
    global _camera, _camera_link, _heartbeat, _robot, _run_ready
#     _burst_cancel.set()  # cancel any running burst
    for obj, method in [(_camera_link, 'unlink'), (_camera, 'stop'), (_robot, 'stop')]:
        try:
            if obj: getattr(obj, method)()
        except Exception:
            pass
    try:
        if _heartbeat: _heartbeat.unobserve_all()
    except Exception:
        pass
    _camera = _camera_link = _heartbeat = _robot = None
    _run_ready = False

cleanup()  # always tear down stale links from any previous run

# ── Robot & Camera Init ───────────────────────────────────────────────────────
if JETBOT_AVAILABLE:
    try:
        _robot = Robot()
    except Exception as e:
        print(f"⚠️  Robot init failed: {e}")

    try:
        _camera = Camera.instance(width=224, height=224)
    except Exception as e:
        print(f"⚠️  Camera init failed: {e}")

# ── Movement Helpers (Option A — re-triggerable bursts) ──────────────────────
# def _burst(left_val, right_val):
#     if _robot is None:
#         return
#     _burst_cancel.set()    # interrupt any currently running burst
#     _burst_cancel.clear()  # reset for the new burst
#     def _run(cancel):
#         try:
#             lv = left_val  if not SWAP_MOTORS else right_val
#             rv = right_val if not SWAP_MOTORS else left_val
#             _robot.left_motor.value  = lv
#             _robot.right_motor.value = rv
#             cancel.wait(timeout=BURST_DURATION)  # interruptible sleep
#         finally:
#             _robot.stop()
#     threading.Thread(target=_run, args=(_burst_cancel,), daemon=True).start()
    
def _burst(left_val, right_val):
    if _robot is None:
        return
    def _run():
        lv = left_val  if not SWAP_MOTORS else right_val
        rv = right_val if not SWAP_MOTORS else left_val
        _robot.left_motor.value  = lv
        _robot.right_motor.value = rv
        time.sleep(BURST_DURATION)
        _robot.stop()
    threading.Thread(target=_run, daemon=True).start()

def move_forward():  
    _burst( BURST_SPEED,  BURST_SPEED)
    time.sleep(BURST_DURATION)
    move_backward()
def move_backward(): _burst(0, 0)
def move_left():     
    _burst(-BURST_SPEED,  BURST_SPEED)
    time.sleep(BURST_DURATION)
    move_backward()
def move_right():    
    _burst( BURST_SPEED, -BURST_SPEED)
    time.sleep(BURST_DURATION)
    move_backward()
def stop_robot():
#     _burst_cancel.set()  # cancel any running burst
    if _robot: _robot.stop()

# ── Dataset Helpers ───────────────────────────────────────────────────────────
def _init_run(run_name: str) -> bool:
    global _run_dirs, _run_ready, _counts
    if not run_name:
        return False
    base = Path('data') / run_name
    _run_dirs = {}
    _counts   = {}
    for label in ('left', 'forward', 'right'):
        p = base / label
        p.mkdir(parents=True, exist_ok=True)
        _run_dirs[label] = p
        _counts[label]   = len(list(p.glob('*.jpg')))  # resume existing count
    _run_ready = True
    return True

def _save_image(label: str):
    global _counts
    if not _run_ready:
        run_name = suffix_input.value.strip()
        if not run_name:
            with status_out:
                print("❌ Enter a run name first.")
            return
        _init_run(run_name)
        for lbl in ('left', 'forward', 'right'):
            count_widgets[lbl].value = _counts[lbl]
        with status_out:
            print(f"📁 Dataset initialized: data/{run_name}/")

    img_bytes = image_widget.value
    if not img_bytes:
        with status_out:
            print("⚠️  No camera frame available — nothing saved.")
        return

    n    = _counts[label]
    name = f"{label}_{suffix_input.value.strip()}_{n:04d}.jpg"
    path = _run_dirs[label] / name
    try:
        path.write_bytes(img_bytes)
        _counts[label] += 1
        count_widgets[label].value = _counts[label]
        with status_out:
            print(f"💾 Saved: {path}")
    except Exception as e:
        with status_out:
            print(f"❌ Save error: {e}")

# ── Camera Preview Helpers ────────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg', width=300, height=300)

def _start_camera_link():
    global _camera_link
    if _camera is None:
        return
    try:
        if _camera_link:
            _camera_link.unlink()
        _camera_link = traitlets.dlink(
            (_camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg
        )
    except Exception as e:
        with status_out:
            print(f"⚠️  Camera link error: {e}")

def _stop_camera_link():
    global _camera_link
    try:
        if _camera_link:
            _camera_link.unlink()
            _camera_link = None
    except Exception:
        pass

if JETBOT_AVAILABLE and _camera:
    _start_camera_link()

# ── Widgets ───────────────────────────────────────────────────────────────────
def _btn(desc, color='#333', w='110px', h='50px'):
    return widgets.Button(
        description=desc,
        layout=widgets.Layout(width=w, height=h),
        style=widgets.ButtonStyle(button_color=color)
    )

# Run name
suffix_input = widgets.Text(
    placeholder='e.g. run1',
    description='Run name:',
    layout=widgets.Layout(width='320px')
)
init_btn = _btn('📂 Init Dataset', '#2a6496', w='150px', h='36px')

# Image counters
count_widgets = {
    lbl: widgets.IntText(value=0, description=lbl, layout=widgets.Layout(width='130px'))
    for lbl in ('left', 'forward', 'right')
}

# Save buttons
save_btns = {
    'left':    _btn('💾 Left',    '#1a6b3a'),
    'forward': _btn('💾 Forward', '#1a6b3a'),
    'right':   _btn('💾 Right',   '#1a6b3a'),
}

# Movement buttons
move_btns = {
    'forward':  _btn('⬆ Forward',  '#2c5f8a'),
    'left':     _btn('⬅ Left',     '#2c5f8a'),
    'stop':     _btn('⏹ STOP',     '#8a2c2c'),
    'right':    _btn('➡ Right',    '#2c5f8a'),
    'backward': _btn('⬇ Backward', '#2c5f8a'),
}

# Speed / duration sliders
speed_slider = widgets.FloatSlider(
    value=BURST_SPEED, min=0.1, max=1.0, step=0.05,
    description='Speed:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)
duration_slider = widgets.FloatSlider(
    value=BURST_DURATION, min=0.05, max=2.0, step=0.05,
    description='Duration:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

# Camera toggle + utility buttons
cam_toggle      = widgets.ToggleButton(value=True, description='📷 Preview ON',
                                       layout=widgets.Layout(width='140px', height='36px'))
restart_cam_btn = _btn('🔄 Restart Cam', '#555', w='140px', h='36px')
shutdown_btn    = _btn('🔴 Shutdown',    '#6b1a1a', w='140px', h='36px')

# Status log — defined BEFORE heartbeat is initialized
status_out = widgets.Output(
    layout=widgets.Layout(height='130px', overflow_y='auto',
                          border='1px solid #444', padding='4px')
)

# ── Heartbeat — must come AFTER status_out is defined ────────────────────────
def _heartbeat_handler(change):
    if JETBOT_AVAILABLE and change['new'] == Heartbeat.Status.dead:
        stop_robot()
        try:
            with status_out:
                print("💀 Heartbeat lost — robot stopped.")
        except NameError:
            pass

if JETBOT_AVAILABLE and _robot:
    try:
        _heartbeat = Heartbeat(period=0.5)
        _heartbeat.observe(_heartbeat_handler, names='status')
    except Exception as e:
        print(f"⚠️  Heartbeat init failed: {e}")

# ── Callbacks ─────────────────────────────────────────────────────────────────
def _on_init(b):
    run_name = suffix_input.value.strip()
    if not run_name:
        with status_out: print("❌ Enter a run name first.")
        return
    _init_run(run_name)
    for lbl in ('left', 'forward', 'right'):
        count_widgets[lbl].value = _counts[lbl]
    with status_out:
        print(f"✅ Ready — data/{run_name}/  (existing counts loaded)")

def _on_speed(change):
    global BURST_SPEED
    BURST_SPEED = change['new']

def _on_duration(change):
    global BURST_DURATION
    BURST_DURATION = change['new']

def _on_cam_toggle(change):
    if change['new']:
        cam_toggle.description = '📷 Preview ON'
        _start_camera_link()
    else:
        cam_toggle.description = '📷 Preview OFF'
        _stop_camera_link()

def _on_restart_cam(b):
    global _camera
    _stop_camera_link()
    try:
        if _camera: _camera.stop()
    except Exception:
        pass
    try:
        _camera = Camera.instance(width=224, height=224)
        _start_camera_link()
        with status_out: print("🔄 Camera restarted.")
    except Exception as e:
        with status_out: print(f"❌ Camera restart failed: {e}")

def _on_shutdown(b):
    _stop_camera_link()
    stop_robot()
    with status_out: print("🔴 Shutdown complete. Safe to close notebook.")

init_btn.on_click(_on_init)
speed_slider.observe(_on_speed, names='value')
duration_slider.observe(_on_duration, names='value')
cam_toggle.observe(_on_cam_toggle, names='value')
restart_cam_btn.on_click(_on_restart_cam)
shutdown_btn.on_click(_on_shutdown)

for lbl, btn in save_btns.items():
    btn.on_click(lambda b, l=lbl: _save_image(l))

move_btns['forward'].on_click( lambda b: move_forward())
move_btns['backward'].on_click(lambda b: move_backward())
move_btns['left'].on_click(    lambda b: move_left())
move_btns['right'].on_click(   lambda b: move_right())
move_btns['stop'].on_click(    lambda b: stop_robot())

print("✅ Cell 1 complete — run Cell 2 to display the UI.")

Error loading module `ublox_gps`: No module named 'serial'
✅ Cell 1 complete — run Cell 2 to display the UI.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Display Interface
# ─────────────────────────────────────────────────────────────────────────────

display(widgets.VBox([

    # ── Dataset init
    widgets.HTML('<b>Dataset</b>'),
    widgets.HBox([suffix_input, init_btn]),
    widgets.HTML('<hr style="margin:6px 0">'),
    
    # ── Camera
    widgets.HTML('<b>Camera Preview</b>'),
    widgets.HBox([cam_toggle, restart_cam_btn, shutdown_btn]),
    image_widget,
    widgets.HTML('<hr style="margin:6px 0">'),
    
    # ── Movement pad
    widgets.HTML('<b>Movement</b>'),
    widgets.VBox([
        move_btns['forward'],
        widgets.HBox([move_btns['left'], move_btns['stop'], move_btns['right']]),
        move_btns['backward'],
    ], layout=widgets.Layout(align_items='center')),
    widgets.HBox([speed_slider, duration_slider]),
    widgets.HTML('<hr style="margin:6px 0">'),

    # ── Image saving
    widgets.HTML('<b>Save Image</b>'),
    widgets.HBox([save_btns['left'], save_btns['forward'], save_btns['right']]),
    widgets.HBox([count_widgets['left'], count_widgets['forward'], count_widgets['right']]),
    widgets.HTML('<hr style="margin:6px 0">'),

    # ── Status log
    widgets.HTML('<b>Status Log</b>'),
    status_out,

]))